In [17]:
from pathlib import Path
from datetime import datetime, timedelta
from itertools import accumulate
import random
random.seed(0)
def date_range_strings(start: str, end: str, fmt="%Y-%m-%d"):
    s = datetime.strptime(start, fmt).date()
    e = datetime.strptime(end, fmt).date()

    cur = s
    out = []
    while cur <= e:
        out.append(cur.strftime(fmt))
        cur += timedelta(days=1)
    return out

In [3]:
inpath = Path("/Data_two/wyw/data/CETC_PRODUCT/merge/*.jsonl")
inpaths = sorted(list(inpath.parent.glob(inpath.name)))

In [7]:
total = 100
date_range = date_range_strings("2025-08-24","2025-08-31")

In [13]:
prob_density = [0.05] + [0.9 / 6] * 6 + [0.05]

In [15]:
prob_cum = list(accumulate(prob_density))

In [16]:
prob_cum

[0.05, 0.2, 0.35, 0.5, 0.65, 0.8, 0.9500000000000001, 1.0]

In [26]:
"2".endswith(2)

TypeError: endswith first arg must be str or a tuple of str, not int

In [31]:
import jsonlines
outpath = Path("/Data_two/wyw/data/CETC_PRODUCT/enhance")
task_name = "task2"
_inpaths = [p for p in inpaths if p.stem.endswith(task_name)]
with jsonlines.open(outpath / f"{task_name}.jsonl", mode = "w") as writer:
    for inpath in _inpaths:
        source = inpath.stem.split("_", maxsplit=1)[0]
        with jsonlines.open(inpath, "r") as reader:
            for raw_obj in reader.iter(skip_empty=True, skip_invalid=True):
                raw_obj["source"] = source
                raw_obj["download_date"] = random.choices(date_range, weights = prob_density)[0]
                writer.write(raw_obj)